# Notebook 04 — Visualization & Dashboard
**SuperStore Sales & Business Performance Analytics System**

This notebook generates all 14 dashboard charts and displays them inline.
The same charts are saved to `data/output/` as PNG files.

### Charts Generated:
1. KPI Summary Card
2. Monthly Sales & Profit Trend
3. Quarterly Sales & Margin Trend
4. Yearly Sales vs Profit
5. Category Performance
6. Sub-Category Profitability
7. Regional Performance (Pie + Bar)
8. Regional Heatmap (Sales by Region × Category)
9. Top 10 Loss-Making Products
10. Discount Impact on Profit Margin
11. Customer Segment Performance
12. Shipping Mode Analysis
13. Bottom 15 States by Profit Margin
14. Sub-Category Return on Sales

In [ ]:
import sys, os

# Detect Databricks
IN_DATABRICKS = 'DATABRICKS_RUNTIME_VERSION' in os.environ

if IN_DATABRICKS:
    import subprocess
    repo_dir = '/tmp/superstore_repo'
    if not os.path.exists(repo_dir):
        subprocess.run(
            ['git', 'clone', '--depth=1',
             'https://github.com/sachintulla/Databrick-Assignment.git', repo_dir],
            check=True, capture_output=True
        )
    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)
    MOUNT_POINT  = '/mnt/superstore'
    PROJECT_ROOT = repo_dir
    print(f'Databricks: repo cloned → {repo_dir}')
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if PROJECT_ROOT not in sys.path:
        sys.path.insert(0, PROJECT_ROOT)
    MOUNT_POINT = None
    print(f'Local: PROJECT_ROOT={PROJECT_ROOT}')

import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from src.analytics.sales_analytics import SalesAnalytics
from src.analytics.profitability_analytics import ProfitabilityAnalytics
from src.visualization.dashboard import Dashboard
from src.utils.config import load_config, get_path
cfg = load_config()

if IN_DATABRICKS:
    parquet_path = f'/dbfs{MOUNT_POINT}/silver/superstore_clean.parquet'
else:
    parquet_path = os.path.join(
        get_path('processed_data_dir', cfg), cfg['data']['processed_file']
    )
clean_df = pd.read_parquet(parquet_path)
print(f'Loaded {len(clean_df):,} rows for dashboard generation')


In [ ]:
sa = SalesAnalytics(clean_df)
pa = ProfitabilityAnalytics(clean_df)
dash = Dashboard(sa, pa)

print('Generating all dashboard charts...')
saved_charts = dash.generate_all()
print(f'\n{len(saved_charts)} charts saved to: data/output/')

## View Generated Charts Inline

In [ ]:
from IPython.display import Image, display
import glob

output_dir = get_path('output_dir', cfg)
chart_files = sorted(glob.glob(os.path.join(output_dir, '*.png')))

for chart_path in chart_files:
    chart_name = os.path.basename(chart_path).replace('.png', '').replace('_', ' ').title()
    print(f'\n### {chart_name}')
    display(Image(filename=chart_path, width=900))

## Databricks Dashboard Notes

To build the **Azure Databricks Dashboard**:

1. Upload this notebook to your Databricks workspace
2. Run all cells to generate charts (they will render inline in Databricks)
3. Click **View → Add to Dashboard** in Databricks UI for each cell
4. Arrange KPI card, trend charts, and category charts in the dashboard layout
5. Set the dashboard to **auto-refresh** with your pipeline schedule

### Recommended Dashboard Layout:
```
┌─────────────────────────────────────────────────────┐
│              KPI Summary Card (full width)           │
├─────────────────────┬───────────────────────────────┤
│  Monthly Sales Trend│   Regional Performance        │
├─────────────────────┼───────────────────────────────┤
│  Category Perf Bar  │   Regional Heatmap             │
├─────────────────────┼───────────────────────────────┤
│  Loss Products      │   Discount Impact              │
├─────────────────────┴───────────────────────────────┤
│          Sub-Category Return on Sales (full)         │
└─────────────────────────────────────────────────────┘
```